# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess

if not os.path.isdir("Flyranks-internship-assignmnet-1"):
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/MaryamNaveed-bioinfo/Flyranks-internship-assignmnet-1"], check=True)
os.chdir("Flyranks-internship-assignmnet-1")
print("Working dir:", os.getcwd())

subprocess.run(["pip", "install", "-q", "duckdb", "scikit-learn"], check=True)

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Ready to query:", REL)

Working dir: /content/Flyranks-internship-assignmnet-1
Connected. Ready to query: hf://datasets/FlyRank/internship-warehouse


In [2]:
!pip install -q pypdf
from pypdf import PdfReader

reader = PdfReader("docs/flyrank-seo-research-march-2026.pdf")
print(f"Pages: {len(reader.pages)}")

full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print(full_text[:3000])  # first ~3000 characters to start

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 6.2 MB/s eta 0:00:00
Pages: 36
DATA REPORT · MARCH 2026
THE STATE OF
AI-DRIVEN SEO
IN NUMBERS.
We analyzed 341,701 content pieces across 57 brands, using 
direct portfolio comparisons as the main evidence base and 
an exploratory ML appendix on 61,790 active content records. 
The goal is a public-safe study that a reader can act on without 
FlyRank tooling.
flyrank.com
FlyRank — The State of AI-Driven SEO, March 2026 1
FlyRank ABSTRACT
About This Study
This paper examines 341,701 content pieces across 57 brands, spanning 469,879,632 search impressions, 1,514,819 
clicks,1,635,404 sessions, and 17,344 AI sessions.
Rather than reproducing dashboard screenshots or internal table views, we tested portfolio-level questions about 
content age, freshness, position, traffic mix, depth, and AI visibility. Direct aggregate comparisons lead the paper. 
Machine-learning pages appear later as exploratory appendix material.
The goal is pract

In [3]:
print(full_text[3000:8000])

0-day performance window (rolling), 30-day trend comparison, local historical monthly series, and 
local active-content feature-vector cuts where noted.
VERIFIED TOTAL VALUE
Content 341,701
Brands 57
Impressions 469,879,632
Clicks 1,514,819
Sessions 1,635,404
AI Sessions 17,344
Metric Windows
90-day performance window (rolling), 30-day trend comparison, local historical monthly series, and local active-content feature-vector cuts 
where noted.
Local Snapshot Rule
This paper uses the local cached snapshot and local derived exports. Some extended cuts come from the local full feature vector, which is 
an active-content subset with impressions_90d > 0 and sessions_90d > 0.
Evidence Standard
Headline findings prioritize direct aggregate comparisons. ML pages are exploratory appendix material and do not override direct portfolio 
evidence.
Interpretation Rule
External SEO beliefs are treated as hypotheses, not proof. When direct portfolio evidence conflicts with industry narratives, the por

In [4]:
print(full_text[13000:18000])

ity, refresh outdated facts and examples, add missing subtopics, and resubmit the strongest 
updates for recrawl.
Expected: Lifts mature pages from roughly 10.7 health to 34.5 and materially improves impression potential.
Measure: Track impressions, clicks, and average position on refreshed mature pages versus comparable untouched pages.
FlyRank — The State of AI-Driven SEO, March 2026 9
FlyRank FINDING #5 — CONFIRMED
Engagement and Visibility Move Together.
High scroll + high engagement = +11.2 health points. Visibility consistency compounds the effect.
SCROLL × ENGAGEMENT COUNT HEALTH AVG IMP AI % POS
high_scroll × high 1.0K 50.7 629 5.75 15.9
high_scroll × mid 3.6K 50.3 5.3K 1.78 14.3
low_scroll × mid 10.7K 49.4 11.8K 1.34 14.2
high_scroll × low 13.6K 46.7 1.1K 1.43 15.2
low_scroll × low 32.9K 39.5 1.7K 1.53 14.2
HEALTH BY VISIBILITY CONSISTENCY (DAYS VISIBLE IN 90)
consistent 47
intermittent 35
moderate 34
sporadic 28
invisible 6
In this portfolio, stronger engagement patterns and 

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #1 : "The Anatomy of Growing Content" (CONFIRMED)**

*Claim:* Growing pages (rising impressions) are 37.6% longer and 20% younger than
declining pages (74,187 rising vs. 45,272 falling pages), and the paper recommends
expanding thin pages and reviewing aging content based on this gap.

*Methodology question:* Where does the "growing vs. declining" label come from, and
does it risk circularity with the features being compared? The paper defines
`trend_direction` from 30-day-vs-previous-30-day impression change (up = >10% growth).
Word count and age are then compared *across* that label. This is a fair, honest
observational comparison  but it's worth asking whether word count and age were
already correlated with impressions at the time the page was created or last measured,
rather than being causes of the growth. In other words: does a page grow *because* it's
longer and younger, or do longer/younger pages simply tend to already be in a growth
phase for other reasons (e.g., recency bias in how new content gets initially promoted
or indexed)? The paper avoids this trap reasonably well by calling it "directionally
robust... observational," but a reader could still walk away thinking "make pages
longer to make them grow," which the data doesn't establish as causal.

**Finding #6 :  "AI Traffic: A Different Signal" (CONFIRMED)**

*Claim:* Pages with high AI-referral traffic average ~9x more impressions than
no-AI pages, despite having a *weaker* average Google position (19.8 vs. 14.2)
suggesting "AI-referral visibility is not simply a mirror of Google rank."

*Methodology question:* Does the sample size support this comparison's strength?
The high-AI bucket is only 873 pages, against 57,100 no-AI pages a roughly 65:1
imbalance. The paper itself flags the proportional caveat (AI is 1.06% of sessions),
which is good practice, but the specific comparison "9x more impressions despite
weaker position" is a strong, quotable claim built on a fairly small high-AI group.
Worth asking: how stable is this 9x figure across different months, or would it shift
substantially with a slightly different bucketing threshold for "high_ai"? The paper
handles this reasonably by keeping the final claim narrow ("real, growing, and
behaviorally different") rather than overclaiming causation but the average-based
comparison on 873 pages deserves a stated confidence interval or a robustness check
before being treated as a stable pattern rather than a snapshot.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before/after: naive random split vs. honest grouped split**

Week 5 only used a client-grouped split. Per the leakage-hunting skill, "random is
rarely honest" a random row split lets the model see other pages from the same
client in both train and test, potentially memorizing client-specific quirks rather
than learning a generalizable pattern. This section rebuilds the naive (random) version
for direct comparison, using the same clean feature set (no `sessions_organic`, removed
in Week 5 for leakage) and same model settings as Week 5.

In [5]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

q_features = f"""
WITH page_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        SUM(sessions_organic) AS sessions_organic,
        SUM(sessions_direct) AS sessions_direct,
        SUM(sessions_referral) AS sessions_referral,
        SUM(sessions_social) AS sessions_social,
        SUM(sessions_paid) AS sessions_paid,
        SUM(sessions_ai) AS sessions_ai,
        SUM(scroll_events) AS scroll_events
    FROM read_parquet('{REL}/fact_content_daily_performance_sample.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    *,
    gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr
FROM page_agg
WHERE gsc_impressions >= 100
"""
df = con.sql(q_features).df()

df['position_bucket'] = pd.cut(
    df['gsc_avg_position'],
    bins=[-0.01, 3, 10, 20, float('inf')],
    labels=['1-3 (top)', '4-10', '11-20', '21+']
)
bucket_avg = df.groupby('position_bucket', observed=True)['ctr'].mean().rename('baseline_pred')
df = df.merge(bucket_avg, on='position_bucket', how='left')

feature_cols_clean = [
    'gsc_avg_position', 'gsc_impressions',
    'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
    'sessions_direct', 'sessions_referral',
    'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events'
]  # sessions_organic excluded — confirmed leaky in Week 5 (r=0.988 with gsc_clicks)

print(f"Rows: {len(df)}, Clients: {df['client_hash_id'].nunique()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101917, Clients: 46


In [7]:
X = df[feature_cols_clean]
y = df['ctr']
groups = df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]
baseline_test_grp = df['baseline_pred'].iloc[test_idx]

model_grouped = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
model_grouped.fit(X_train_grp, y_train_grp)
grouped_preds = model_grouped.predict(X_test_grp)

baseline_mae = mean_absolute_error(y_test_grp, baseline_test_grp)
grouped_mae = mean_absolute_error(y_test_grp, grouped_preds)

print(f"Baseline MAE: {baseline_mae:.6f}")
print(f"Honest (grouped) model MAE: {grouped_mae:.6f}")

Baseline MAE: 0.003729
Honest (grouped) model MAE: 0.003337


In [8]:
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    df[feature_cols_clean], df['ctr'], test_size=0.2, random_state=42
)

model_naive = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
model_naive.fit(X_train_naive, y_train_naive)
naive_preds = model_naive.predict(X_test_naive)

naive_mae = mean_absolute_error(y_test_naive, naive_preds)

split_comparison = pd.DataFrame({
    'Split type': ['Naive (random rows)', 'Honest (grouped by client)'],
    'MAE': [naive_mae, grouped_mae],
    'Test set size': [len(X_test_naive), len(X_test_grp)]
})
print(split_comparison)

                   Split type       MAE  Test set size
0         Naive (random rows)  0.002902          20384
1  Honest (grouped by client)  0.003337          32285


**Result: the naive split looks better than the honest one : this gap is the finding.**

| Split type | MAE | Test rows |
|---|---|---|
| Naive (random rows) | 0.002902 | 20,384 |
| Honest (grouped by client) | 0.003337 | 32,285 |

The naive random split shows a lower (better-looking) error than the honest
client-grouped split a gap of about 0.000435, roughly **15% relative difference**.
This matches exactly what the leakage-hunting skill predicts: a random split lets the
model see other pages from the *same* clients in both train and test, so it partly
memorizes client-specific patterns rather than learning something that generalizes to
entirely new clients. The grouped-split number (0.003337) is the one that should be
trusted and reported  it reflects performance on 10 clients the model never saw during
training, which is the realistic deployment scenario. The 15% gap between the two is
itself evidence of how much apparent "skill" was actually memorization.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage audit : final feature set**

Following the attack checklist from `hunting-leakage-and-validating`:
- Timeline: all features are aggregated over the same month as the label (`ctr`)
  this is acceptable here since the goal is describing current-state CTR, not
  predicting a *future* outcome from past data. No future window is used.
- Label-derived features: `sessions_organic` was tested and removed in Week 5
  (correlation 0.988 with `gsc_clicks`, part of the CTR calculation itself).
- Product flags: no FlyRank-generated flags (e.g. `optimization_eligible_date`) are
  used anywhere in the feature set confirmed in Week 4's leakage check and
  unchanged since.
- Split: grouped by `client_hash_id` (Section 2), not random rows.
- Base rate: printed alongside every MAE comparison (Section 2's table).

In [9]:
# Attack check: re-add sessions_organic and confirm the score jumps —
# proving our test harness actually detects leakage, not just assumes it.
feature_cols_with_leak = feature_cols_clean + ['sessions_organic']

X_leak = df[feature_cols_with_leak]
X_train_leak, X_test_leak = X_leak.iloc[train_idx], X_leak.iloc[test_idx]

model_leak = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
model_leak.fit(X_train_leak, y_train_grp)
leak_preds = model_leak.predict(X_test_leak)
leak_mae = mean_absolute_error(y_test_grp, leak_preds)

leakage_check = pd.DataFrame({
    'Feature set': ['Clean (final)', 'With sessions_organic (leaky)'],
    'MAE': [grouped_mae, leak_mae]
})
print(leakage_check)
print(f"\nCorrelation check: sessions_organic vs gsc_clicks = {df[['sessions_organic','gsc_clicks']].corr().iloc[0,1]:.3f}")

                     Feature set       MAE
0                  Clean (final)  0.003337
1  With sessions_organic (leaky)  0.002837

Correlation check: sessions_organic vs gsc_clicks = 0.988


**Result: the attack check confirms our leakage detection actually works.**

| Feature set | MAE |
|---|---|
| Clean (final) | 0.003337 |
| With `sessions_organic` (leaky) | 0.002837 |

Re-adding the known-leaky feature (`sessions_organic`, r=0.988 with `gsc_clicks`)
measurably improves the score (0.003337 -> 0.002837), exactly as the skill's "how to
verify" step expects: deliberately adding a leaky feature should make the test harness
show improvement, confirming the harness can actually detect leakage rather than just
assuming it. Since the improvement disappears once the feature is removed, this
confirms the clean feature set (Section 2's result, MAE 0.003337) is the honest number
to report the earlier ~24% improvement claimed in Week 5 before this feature was
caught was inflated by leakage, and the real, defensible improvement over baseline is
smaller: (0.003729 − 0.003337) / 0.003729 ≈ **10.5%**.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest original claim (from Week 5, before the leakage catch):**

> "Random Forest achieves MAE 0.00284 vs. the baseline's 0.00373 a ~24% error
> reduction. The model generalizes to clients it never saw during training."

**Why this needs rewriting:** it states a causal-sounding, confident "24% error
reduction" as if fully earned by the model's real skill, when a meaningful share of
that number came from a leaky feature (`sessions_organic`) that was a near-duplicate of
the target. It also doesn't acknowledge the naive-vs-grouped split gap found in
Section 2.

**Rewritten in safe language:**
> "On a client-grouped split testing on clients the model never saw during
> training the model's predictions were observed to have a lower mean absolute
> error than the position-bucket baseline (0.00334 vs. 0.00373), a directional
> improvement of roughly 10.5%. This is offered as decision-support for prioritizing
> review candidates, not as a claim of causal understanding of what drives CTR. A
> known leakage risk (`sessions_organic`) was identified and removed before this
> figure was reported, and a naive (non-grouped) split was measured to overstate
> performance by roughly 15% relative to the honest split a reminder that
> validation design materially changes what a score means."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.